# Talking to models with the OpenAI-style chat format

This notebook shows the **request and response format** that WebLLM uses for chat. The shape is almost the same as the format used by OpenAI's Chat Completions API, which is why the same ideas carry over to many other AI tools.

Everything here runs in your browser.

## Step 1: Get set up

This checks for WebGPU, loads WebLLM, and defines a helper for turning Python dictionaries into JavaScript objects.

In [ ]:
import js
from pyodide.ffi import to_js, create_proxy
from IPython.display import display
from pprint import pprint

if getattr(js.navigator, "gpu", None):
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")

engine = None
loaded_model = None


def js_obj(d):
    """Turn a Python dict into a JavaScript object."""
    return to_js(d, dict_converter=js.Object.fromEntries)


def progress_bar():
    """Show a progress bar that updates in place."""
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    return status, create_proxy(show)


async def start_model(model_id):
    """Load a model, downloading it first if needed."""
    global engine
    status, progress = progress_bar()

    async def attempt(chat_options=None):
        global engine
        if engine is None:
            if chat_options:
                engine = await webllm.CreateMLCEngine(
                    model_id, js_obj({"initProgressCallback": progress}), js_obj(chat_options)
                )
            else:
                engine = await webllm.CreateMLCEngine(
                    model_id, js_obj({"initProgressCallback": progress})
                )
        else:
            engine.setInitProgressCallback(progress)
            if chat_options:
                await engine.reload(model_id, js_obj(chat_options))
            else:
                await engine.reload(model_id)

    try:
        await attempt()
    except Exception as error:
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            await attempt({"sliding_window_size": -1})
        else:
            raise


## Step 2: Choose and load a model

This notebook is about the chat format, so we will just pick one recommended chat model and load it. If it is not already saved in your browser, WebLLM will download it first.

In [ ]:
MODEL_ID = "gemma3-1b-it-q4f16_1-MLC"

print(f"Model to load: {MODEL_ID}")
if await webllm.hasModelInCache(MODEL_ID):
    print("This model is already saved in your browser.")
else:
    print("This model is not saved yet, so the next step will download it.")

In [ ]:
await start_model(MODEL_ID)
loaded_model = MODEL_ID
print(f"{loaded_model} is loaded and ready.")

## Step 3: Build the smallest possible chat request

A chat request starts with a **list of messages**. Each message has a role and some content.

The three main roles are:

- `system`: instructions that set the model's behavior
- `user`: what you ask
- `assistant`: earlier replies from the model

We start with just one user message.

In [ ]:
messages = [
    {"role": "user", "content": "Explain what a token is in two short sentences."}
]

messages

## Step 4: Add the rest of the request

Along with `messages`, we can send settings that affect how the model replies. We will use a few safe defaults that help small models avoid repetition.

In [ ]:
request_dict = {
    "messages": messages,
    "temperature": 0.7,
    "max_tokens": 300,
    "frequency_penalty": 0.5,
}

request_dict

This is still a normal Python dictionary. WebLLM runs in JavaScript, so we convert it before sending it.

In [ ]:
request = js_obj(request_dict)
print("Request is ready to send.")

## Step 5: Send the request

The next cell sends the request and saves the whole reply object.

In [ ]:
if loaded_model is None:
    print("No model is loaded yet. Run Step 2.")
else:
    reply = await engine.chat.completions.create(request)
    print("Got a reply.")

## Step 6: Look at the full response JSON

The reply contains the answer, some bookkeeping fields, and token counts.

In [ ]:
if loaded_model is None:
    print("No model is loaded yet. Run Step 2.")
else:
    pprint(reply.to_py())

## Step 7: Pull out the most important fields

The most useful parts are usually:

- `choices[0].message.content`: the answer text
- `choices[0].finish_reason`: why the model stopped
- `usage.prompt_tokens`: how many input tokens you sent
- `usage.completion_tokens`: how many output tokens the model wrote
- `usage.extra.prefill_tokens_per_s` and `usage.extra.decode_tokens_per_s`: rough speed numbers when WebLLM reports them

In [ ]:
if loaded_model is None:
    print("No model is loaded yet. Run Step 2.")
else:
    response = reply.to_py()
    print("Answer:")
    print(response['choices'][0]['message']['content'])
    print()
    print("finish_reason:", response['choices'][0].get('finish_reason'))
    usage = response.get('usage', {})
    print("prompt_tokens:", usage.get('prompt_tokens'))
    print("completion_tokens:", usage.get('completion_tokens'))
    print("total_tokens:", usage.get('total_tokens'))
    extra = usage.get('extra', {}) or {}
    print("prefill_tokens_per_s:", extra.get('prefill_tokens_per_s'))
    print("decode_tokens_per_s:", extra.get('decode_tokens_per_s'))

## Step 8: Add a system message

A `system` message sets the overall behavior. It does not guarantee perfect obedience, but it often helps.

In [ ]:
system_messages = [
    {"role": "system", "content": "You are a patient teacher for beginners."},
    {"role": "user", "content": "Explain what a token is in two short sentences."},
]

system_messages

## Step 9: Add earlier assistant and user turns

A multi-turn conversation is just a longer `messages` list. You keep appending what the user said and what the assistant said. The model does not remember anything outside the list you send.

In [ ]:
conversation_messages = [
    {"role": "system", "content": "You are a patient teacher for beginners."},
    {"role": "user", "content": "My name is Maya."},
    {"role": "assistant", "content": "Nice to meet you, Maya."},
    {"role": "user", "content": "What name did I just tell you?"},
]

conversation_messages

## Step 10: Make a reusable helper

Now that you have seen the raw format, here is a helper that sends any message list plus any extra settings.

In [ ]:
async def chat_once(messages, **settings):
    if loaded_model is None:
        return {"error": "No model is loaded yet. Run Step 2."}

    defaults = {
        "messages": messages,
        "temperature": 0.7,
        "max_tokens": 300,
        "frequency_penalty": 0.5,
    }
    defaults.update(settings)
    reply = await engine.chat.completions.create(js_obj(defaults))
    return reply.to_py()

## Step 11: Try a multi-turn chat

Because the whole conversation is inside `messages`, the same helper works for a one-turn request or a longer exchange.

In [ ]:
response = await chat_once(conversation_messages)
if 'error' in response:
    print(response['error'])
else:
    print(response['choices'][0]['message']['content'])

## Step 12: Experiment with the main settings

The next sections each change one setting at a time. Run the cells a few times and compare the outputs.

### 12a: Temperature

`temperature` controls randomness. Low values usually give safer, more repeatable answers. Higher values usually give more variety.

In [ ]:
prompt = [{"role": "user", "content": "Write a one-line slogan for a coffee shop."}]

for value in [0.0, 0.7, 1.2]:
    response = await chat_once(prompt, temperature=value)
    if 'error' in response:
        print(response['error'])
        break
    print(f"temperature={value}: {response['choices'][0]['message']['content']}\n")

### 12b: top_p

`top_p` is another way to control randomness. It limits the model to a smaller or larger slice of the likely next tokens.

In [ ]:
for value in [0.3, 0.7, 1.0]:
    response = await chat_once(prompt, top_p=value)
    if 'error' in response:
        print(response['error'])
        break
    print(f"top_p={value}: {response['choices'][0]['message']['content']}\n")

### 12c: max_tokens

`max_tokens` sets an upper limit on how long the answer can be. If the model hits the limit, `finish_reason` is often `"length"`.

In [ ]:
for value in [20, 60, 150]:
    response = await chat_once(
        [{"role": "user", "content": "List five study tips for a hard class."}],
        max_tokens=value,
    )
    if 'error' in response:
        print(response['error'])
        break
    print(f"max_tokens={value}, finish_reason={response['choices'][0].get('finish_reason')}")
    print(response['choices'][0]['message']['content'])
    print()

### 12d: frequency_penalty

`frequency_penalty` pushes the model away from repeating the same words again and again. This is especially useful with small models.

In [ ]:
for value in [0.0, 0.5, 1.0]:
    response = await chat_once(
        [{"role": "user", "content": "Write a short chant about bicycles."}],
        frequency_penalty=value,
        max_tokens=80,
    )
    if 'error' in response:
        print(response['error'])
        break
    print(f"frequency_penalty={value}: {response['choices'][0]['message']['content']}\n")

### 12e: presence_penalty

`presence_penalty` nudges the model toward bringing in new topics or words instead of staying too close to what it has already said.

In [ ]:
for value in [0.0, 0.8, 1.5]:
    response = await chat_once(
        [{"role": "user", "content": "Give me ideas for a school club slogan."}],
        presence_penalty=value,
        max_tokens=60,
    )
    if 'error' in response:
        print(response['error'])
        break
    print(f"presence_penalty={value}: {response['choices'][0]['message']['content']}\n")

### 12f: seed

`seed` asks the model to be more repeatable. It is helpful for demos and experiments, though exact repeatability is not always perfect across systems.

In [ ]:
for value in [7, 7, 21]:
    response = await chat_once(
        [{"role": "user", "content": "Write a two-line welcome message for a math club."}],
        seed=value,
    )
    if 'error' in response:
        print(response['error'])
        break
    print(f"seed={value}: {response['choices'][0]['message']['content']}\n")

### 12g: stop

`stop` tells the model to stop when it reaches a marker you choose.

In [ ]:
response = await chat_once(
    [{"role": "user", "content": "Write three short lines about the moon. End each line with ###"}],
    stop=["###"],
    max_tokens=80,
)
if 'error' in response:
    print(response['error'])
else:
    print(response['choices'][0]['message']['content'])
    print("finish_reason:", response['choices'][0].get('finish_reason'))

## Step 13: The same shape also works with cloud models

The important idea is the **shape** of the request and response. If you later use a cloud model, you will still send a `messages` list plus settings like `temperature`, and you will still get back `choices`, `message.content`, `finish_reason`, and `usage`.

WebLLM changes **where** the model runs. The data format stays familiar.

## Your turn

Change the messages and settings above. Try to make the model sound more formal, more playful, or more repetitive, and notice which setting matters most.